# Ordered Logistic Regression Results (FAIR²) Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring the FAIR² rangeland knowledge adoption regression dataset using the `mlcroissant` library.

### Dataset Source
Source Croissant schema: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {getattr(metadata, 'name', '')}\n")
print(f"Description: {getattr(metadata, 'description', '')}\n")

## 2. Data Overview
Review available record sets and fields (by their `@id`).

In [ ]:
# List all record sets with their @id and names
record_sets = dataset.record_sets
print("Record sets and their @ids:")
for rs in record_sets:
    print(f"- @id: {rs.id}, name: {getattr(rs, 'name', '')}")

# For demonstration, list fields in each record set
for rs in record_sets:
    print(f"\nRecord set @id: {rs.id}, name: {getattr(rs, 'name', '')}")
    try:
        fields = rs.fields
        for f in fields:
            print(f"  Field @id: {f.id}, name: {getattr(f, 'name', '')}")
    except AttributeError:
        print("  No fields available.")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Reference each entity (record set, field) by its `@id`.

In [ ]:
# Collect data from all record sets by their @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for record set: {rs_id} with columns: {df.columns.tolist()}")
    else:
        print(f"No records for record set {rs_id}")

# Display head of each DataFrame (if not too many)
for rs_id, df in dataframes.items():
    print(f"\nData sample for record set: {rs_id}")
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common EDA: filter by numeric field, normalize values, group by a categorical field, all by `@id`.

*Replace `<record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with concrete `@id` values from section 2 above according to your analysis interest!*

In [ ]:
# --- Please update these @ids based on the overview above!
# For example purposes, example IDs are used; replace with real @ids in actual analysis.
example_record_set_id = next(iter(dataframes.keys())) if dataframes else None
example_numeric_field_id = None
# Guess a numeric column (first float/int column)
if example_record_set_id is not None:
    df = dataframes[example_record_set_id]
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            example_numeric_field_id = col
            break

# Guess a group field ID (first object or category column)
example_group_field_id = None
if example_record_set_id is not None:
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]) and col != example_numeric_field_id:
            example_group_field_id = col
            break

if example_record_set_id and example_numeric_field_id:
    print(f"Using @id for record set: {example_record_set_id}")
    print(f"Using numeric field @id: {example_numeric_field_id}")
    filtered_df = df[df[example_numeric_field_id] > 0]  # Example threshold: > 0
    print(f"Filtered records with '{example_numeric_field_id}' > 0:")
    display(filtered_df.head())

    # Normalize numeric field
    norm_col = f"{example_numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[example_numeric_field_id] - filtered_df[example_numeric_field_id].mean()) / filtered_df[example_numeric_field_id].std()
    print(f"Normalized '{example_numeric_field_id}' for filtered records:")
    display(filtered_df[[example_numeric_field_id, norm_col]].head())

    # Grouping if group field is available
    if example_group_field_id:
        print(f"Grouping by field with @id: {example_group_field_id}")
        grouped_df = filtered_df.groupby(example_group_field_id).mean(numeric_only=True)
        print(f"Grouped mean data:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields. All code below references columns by their `@id` as in previous steps.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: histogram of the selected numeric field
if example_record_set_id and example_numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[example_numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of '{example_numeric_field_id}'")
    plt.xlabel(example_numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

# Optional: barplot grouped by a categorical field
if example_group_field_id and example_numeric_field_id:
    plt.figure(figsize=(10, 5))
    grouped = df.groupby(example_group_field_id)[example_numeric_field_id].mean().sort_values()
    grouped.plot(kind='bar')
    plt.title(f"Average {example_numeric_field_id} by {example_group_field_id}")
    plt.xlabel(example_group_field_id)
    plt.ylabel(f"Mean {example_numeric_field_id}")
    plt.tight_layout()
    plt.show()


## 6. Conclusion
This notebook demonstrates loading, overview, and analysis of a Croissant-conformant dataset using `mlcroissant` and referencing all data elements by `@id`. You can extend it for in-depth FAIR² rangeland policy and regression analysis.